In [ ]:
!pip install -U openai-whisper
!apt-get update -qq
!apt-get install -y ffmpeg -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=55dfb50d8db17836e108971cc40a0ab18f2381726f6900e6b6df2fa07002551c
  Stored in directory: /root/.cache/pip/wheels/ca/58/d5/fb4539ad74c3ca81eb40f7eda020ac77d080b33ad57449d485
Successfully built openai-whisper
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
import whisper

whisper_model = whisper.load_model("base", device=device)

print("Whisper loaded successfully!")
print("Device:", whisper_model.device)

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 88.4MiB/s]


Whisper loaded successfully!
Device: cuda:0


In [ ]:
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode

def record_audio(filename="test_english.webm", duration=10):

    js = Javascript("""
    async function recordAudio(duration) {
        const stream = await navigator.mediaDevices.getUserMedia({audio: true});
        const recorder = new MediaRecorder(stream);
        const chunks = [];

        recorder.ondataavailable = event => {
            if (event.data.size > 0) {
                chunks.push(event.data);
            }
        };

        recorder.start();

        await new Promise(resolve =>
            setTimeout(resolve, duration * 1000)
        );

        recorder.stop();

        await new Promise(resolve => {
            recorder.onstop = resolve;
        });

        stream.getTracks().forEach(track => track.stop());

        const blob = new Blob(chunks, {type: "audio/webm"});
        const reader = new FileReader();

        reader.readAsDataURL(blob);

        await new Promise(resolve => {
            reader.onloadend = resolve;
        });

        google.colab.kernel.invokeFunction(
            "notebook.save_audio",
            [reader.result],
            {}
        );
    }

    recordAudio(10);
    """)

    display(js)

    def save_audio(data):
        audio_data = b64decode(data.split(",")[1])

        with open(filename, "wb") as f:
            f.write(audio_data)

        print("Audio saved:", filename)

    output.register_callback(
        "notebook.save_audio",
        save_audio
    )

record_audio()

<IPython.core.display.Javascript object>

Audio saved: test_english.webm


In [ ]:
result = whisper_model.transcribe(
    audio_path,
    language="en",
    task="transcribe"
)

english_text = result["text"].strip()

print("English transcription:")
print(english_text)

NameError: name 'audio_path' is not defined

Temporary


KEEP

In [ ]:
from transformers import MarianTokenizer, MarianMTModel

EXP2_CHECKPOINT = (
    "/content/drive/MyDrive/"
    "Multilingual_Speech_Translator/"
    "experiment_2_final_checkpoints/"
    "checkpoint-12500"
)

translator_tokenizer = MarianTokenizer.from_pretrained(
    EXP2_CHECKPOINT
)

translator_model = MarianMTModel.from_pretrained(
    EXP2_CHECKPOINT
)

translator_model = translator_model.to(device)
translator_model.eval()

print("Experiment 2 checkpoint loaded successfully!")
print("Checkpoint:", EXP2_CHECKPOINT)
print("Device:", device)

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Experiment 2 checkpoint loaded successfully!
Checkpoint: /content/drive/MyDrive/Multilingual_Speech_Translator/experiment_2_final_checkpoints/checkpoint-12500
Device: cuda


In [ ]:
text = "Hello, I am testing my English to Hindi speech translation."

inputs = translator_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128
).to(device)

with torch.no_grad():
    output_ids = translator_model.generate(
        **inputs,
        max_length=128,
        num_beams=4
    )

hindi_text = translator_tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("English:", text)
print("Hindi:", hindi_text)

English: Hello, I am testing my English to Hindi speech translation.
Hindi: नमस्कार, मैं हिन्दी भाषा अनुवाद के लिए अपनी अंग्रेजी परीक्षा का परीक्षण कर रहा हूं।


In [ ]:
def speech_to_hindi(audio_path):

    # -------------------------
    # 1. Speech → English text
    # -------------------------
    result = whisper_model.transcribe(
        audio_path,
        language="en",
        task="transcribe"
    )

    english_text = result["text"].strip()

    # -------------------------
    # 2. English → Hindi
    # -------------------------
    inputs = translator_tokenizer(
        english_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        output_ids = translator_model.generate(
            **inputs,
            max_length=128,
            num_beams=4
        )

    hindi_text = translator_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return english_text, hindi_text

In [ ]:
english_text, hindi_text = speech_to_hindi(
    "/content/test_english.webm"
)

print("=" * 60)
print("English transcription:")
print(english_text)

print("\nHindi translation:")
print(hindi_text)
print("=" * 60)

English transcription:
My name is Ayahas.

Hindi translation:
मेरा नाम असम है।


In [ ]:
!pip install -q gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.28.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [ ]:
from gtts import gTTS

tts = gTTS(
    text=hindi_text,
    lang="hi"
)

output_audio = "/content/hindi_output.mp3"

tts.save(output_audio)

print("Hindi speech generated successfully!")
print("Output:", output_audio)

Hindi speech generated successfully!
Output: /content/hindi_output.mp3


In [ ]:
from IPython.display import Audio, display

display(Audio(output_audio))

Testing

In [ ]:
def english_speech_to_hindi_speech(audio_path):

    # 1. Speech → English text
    result = whisper_model.transcribe(
        audio_path,
        language="en",
        task="transcribe"
    )

    english_text = result["text"].strip()

    # 2. English text → Hindi text
    inputs = translator_tokenizer(
        english_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        output_ids = translator_model.generate(
            **inputs,
            max_length=128,
            num_beams=4
        )

    hindi_text = translator_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    # 3. Hindi text → Hindi speech
    from gtts import gTTS

    output_audio = "/content/hindi_translation.mp3"

    tts = gTTS(
        text=hindi_text,
        lang="hi"
    )

    tts.save(output_audio)

    return english_text, hindi_text, output_audio

In [ ]:
english_text, hindi_text, hindi_audio = english_speech_to_hindi_speech(
    "/content/test_english.webm"
)

print("=" * 60)
print("English transcription:")
print(english_text)

print("\nHindi translation:")
print(hindi_text)

print("\nHindi audio:")
print(hindi_audio)

print("=" * 60)

English transcription:
My name is Ayahas.

Hindi translation:
मेरा नाम असम है।

Hindi audio:
/content/hindi_translation.mp3


In [ ]:
from IPython.display import Audio, display

display(Audio(hindi_audio))

# Interface

In [ ]:
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.5.0 which is incompatible.


In [ ]:
import gradio as gr
from gtts import gTTS
import torch

def translate_speech(audio_path):

    if audio_path is None:
        return "", "", None

    # -------------------------
    # 1. Speech → English text
    # -------------------------
    result = whisper_model.transcribe(
        audio_path,
        language="en",
        task="transcribe"
    )

    english_text = result["text"].strip()

    if not english_text:
        return "", "", None

    # -------------------------
    # 2. English → Hindi
    # -------------------------
    inputs = translator_tokenizer(
        english_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        output_ids = translator_model.generate(
            **inputs,
            max_length=128,
            num_beams=4
        )

    hindi_text = translator_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    # -------------------------
    # 3. Hindi text → Hindi speech
    # -------------------------
    output_audio = "/content/hindi_translation.mp3"

    tts = gTTS(
        text=hindi_text,
        lang="hi"
    )

    tts.save(output_audio)

    return english_text, hindi_text, output_audio

In [ ]:
interface = gr.Interface(
    fn=translate_speech,

    inputs=gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="🎤 Speak English"
    ),

    outputs=[
        gr.Textbox(
            label="English Transcription"
        ),
        gr.Textbox(
            label="Hindi Translation"
        ),
        gr.Audio(
            label="🔊 Hindi Speech"
        )
    ],

    title="English → Hindi Speech Translator",

    description=(
        "Speak in English and the system will "
        "transcribe, translate to Hindi, and generate Hindi speech."
    )
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://81896e3c94c6a7e768.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
